In [ ]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Load dataset
data = pd.read_csv("../Dataset/smart_shoe_dataset_clean.csv")

# ----- STEP 1: Smooth out the Sensor Jitter FIRST -----
data['distance_smooth'] = data['Distance'].rolling(window=3).mean()
data = data.dropna(subset=['distance_smooth'])

# ----- STEP 2: Calculate the Current State -----
def fix_labels(distance):
    if distance < 20:
        return 'danger'
    elif distance >= 20 and distance <= 60:
        return 'warning'
    else:
        return 'safe'

data['Current_State'] = data['distance_smooth'].apply(fix_labels)

# ----- STEP 3: THE FIX (Predict the Future) -----
# Shift the target column up by 3 rows. 
# This forces the model to use CURRENT data to predict what happens NEXT.
data['Target_Future_State'] = data['Current_State'].shift(-3)

# ----- STEP 4: Calculate Speed -----
data['distance_diff'] = data['distance_smooth'].diff()
data['time_diff'] = data['Time'].diff()
data['speed'] = data['distance_diff'] / data['time_diff']

# Clean up all the NaN rows created by shifting and rolling
data = data.dropna()
data = data[(data['speed'] < 1) & (data['speed'] > -1)]

# ----- STEP 5: Features & Split -----
# Using current distance and speed to predict the FUTURE state
X = data[['distance_smooth', 'speed']]
y = data['Target_Future_State'] 

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ----- STEP 6: Train & Evaluate -----
model = RandomForestClassifier(random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("--- REALISTIC PREDICTIVE MODEL ACCURACY ---")
print(classification_report(y_test, predictions))

# Export the trained Random Forest model
joblib.dump(model, 'smart_shoe_model.pkl')
print("\nModel saved successfully as smart_shoe_model.pkl")

--- NEW MODEL ACCURACY ---
              precision    recall  f1-score   support

      danger       0.94      0.94      0.94        16
        safe       1.00      1.00      1.00        45
     warning       0.99      0.99      0.99       174

    accuracy                           0.99       235
   macro avg       0.98      0.98      0.98       235
weighted avg       0.99      0.99      0.99       235



['smart_shoe_model.pkl']